# Inspect Retrieval Evaluation Results

Load retrieval results (formula match or RI match) from CSV and HDF5 files.
- **Overview table**: per-molecule ranks, similarity scores, number of candidates
- **Molecule inspector**: ground truth vs predicted spectrum, decoys, individual similarities
- **Summary table**: top-k accuracies, MRR, mean/median rank

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

try:
    from icicle.utils.visualization import set_style, palette

    set_style()
except ImportError:
    palette = None
    print("icicle visualization utils not available, using defaults")

## Configuration

Set paths to the CSV results file and the corresponding HDF5 spectra file.

In [ ]:
# Set these paths to your evaluation results

PUBCHEM_EVAL_DIR = (
    "/home/magled/icicle-dev/tmp/outputs/results/pubchem_retrieval_eval copy"
)
RI_TYPE = "StdNP"  # matches ri_type in default.yaml; change if you ran a different RI type

CSV_PATH = f"{PUBCHEM_EVAL_DIR}/retrieval_with_ri_{RI_TYPE}_results.csv"
HDF5_PATH = f"{PUBCHEM_EVAL_DIR}/retrieval_with_ri_{RI_TYPE}_spectra.hdf5"

# Similarity metric used for primary ranking in the overview table
PRIMARY_RANKING_METRIC = "cosine_similarity"  # e.g. "cosine_similarity", "entropy_similarity", "weighted_cosine_nist_gc"

# All similarity columns expected in the CSV (auto-detected if left empty)
SIMILARITY_COLUMNS = []  # e.g. ["cosine_similarity", "entropy_similarity", "weighted_cosine_nist_gc"]

# Top-k values for summary table
K_VALUES = [1, 2, 3, 4, 5, 10, 15, 20, 50]

## Load data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Unique targets: {df['spec'].nunique()}")
print(f"Columns: {df.columns.tolist()}")

# Auto-detect similarity columns if not specified
if not SIMILARITY_COLUMNS:
    SIMILARITY_COLUMNS = [c for c in df.columns if c.startswith("rank_")]
    SIMILARITY_COLUMNS = [c.replace("rank_", "") for c in SIMILARITY_COLUMNS]
    print(f"Auto-detected similarity columns: {SIMILARITY_COLUMNS}")

# Check HDF5
hdf5_available = Path(HDF5_PATH).exists()
if hdf5_available:
    with h5py.File(HDF5_PATH, "r") as hf:
        n_targets_hdf5 = len([k for k in hf.keys() if k != "mz_bins"])
        has_mz_bins = "mz_bins" in hf
    print(
        f"HDF5 loaded: {n_targets_hdf5} targets, mz_bins={'yes' if has_mz_bins else 'no'}"
    )
else:
    print(
        f"HDF5 not found at {HDF5_PATH} - spectrum inspection will be unavailable"
    )

## Per-molecule overview

Shows for each target molecule: the rank of the correct answer, its similarity score, the number of candidates, and the best decoy's similarity.

In [ ]:
def build_overview_table(df, similarity_columns, primary_metric=None):
    """Build a per-molecule overview DataFrame.

     Parameters
    -------
     df : pd.DataFrame
         Raw retrieval results CSV.
     similarity_columns : list[str]
         Similarity column names (without 'rank_' prefix).
     primary_metric : str, optional
         Metric to use for sorting. Defaults to first in list.

     Returns
    ----
     pd.DataFrame
         One row per target molecule.
    """
    if primary_metric is None:
        primary_metric = similarity_columns[0]

    rows = []
    for spec_id, grp in df.groupby("spec"):
        correct = grp[~grp["is_decoy"]]
        decoys = grp[grp["is_decoy"]]

        row = {
            "spec": spec_id,
            "n_candidates": len(grp),
            "n_decoys": len(decoys),
        }

        # Correct answer SMILES
        if len(correct) > 0:
            row["correct_smiles"] = correct.iloc[0]["smiles"]
        else:
            row["correct_smiles"] = "N/A"

        # Ranks and scores for each metric
        for sim_col in similarity_columns:
            rank_col = f"rank_{sim_col}"
            if rank_col in grp.columns and len(correct) > 0:
                row[f"rank_{sim_col}"] = int(correct[rank_col].min())
                row[f"correct_{sim_col}"] = float(correct[sim_col].max())
            else:
                row[f"rank_{sim_col}"] = np.nan
                row[f"correct_{sim_col}"] = np.nan

            # Best decoy score
            if sim_col in decoys.columns and len(decoys) > 0:
                row[f"best_decoy_{sim_col}"] = float(decoys[sim_col].max())
            else:
                row[f"best_decoy_{sim_col}"] = np.nan

        rows.append(row)

    overview = pd.DataFrame(rows)
    # Sort by primary metric rank
    primary_rank_col = f"rank_{primary_metric}"
    if primary_rank_col in overview.columns:
        overview = overview.sort_values(
            primary_rank_col, ascending=True
        ).reset_index(drop=True)
    return overview


df_overview = build_overview_table(
    df, SIMILARITY_COLUMNS, PRIMARY_RANKING_METRIC
)
print(f"Overview: {len(df_overview)} target molecules")
display(df_overview.head(20))

In [ ]:
# Quick rank distribution
primary_rank_col = f"rank_{PRIMARY_RANKING_METRIC}"
if primary_rank_col in df_overview.columns:
    ranks = df_overview[primary_rank_col].dropna()
    print(f"Rank statistics ({PRIMARY_RANKING_METRIC}):")
    print(f"  Mean rank:   {ranks.mean():.2f}")
    print(f"  Median rank: {ranks.median():.1f}")
    print(f"  Top-1:       {(ranks <= 1).mean() * 100:.1f}%")
    print(f"  Top-5:       {(ranks <= 5).mean() * 100:.1f}%")
    print(f"  Top-10:      {(ranks <= 10).mean() * 100:.1f}%")
    print(f"  MRR:         {(1.0 / ranks).mean():.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].hist(ranks, bins=50, edgecolor="black", linewidth=0.5)
    axes[0].set_xlabel("Rank of correct answer")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"Rank distribution ({PRIMARY_RANKING_METRIC})")

    # Cumulative accuracy
    max_k = min(int(ranks.max()), 50)
    ks = np.arange(1, max_k + 1)
    accs = [(ranks <= k).mean() * 100 for k in ks]
    axes[1].plot(ks, accs, linewidth=1.5)
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Top-k Accuracy (%)")
    axes[1].set_title("Cumulative top-k accuracy")
    axes[1].set_ylim(0, 105)
    plt.tight_layout()
    plt.show()

## Molecule inspector

Select a target molecule to inspect:
- Ground truth (experimental) spectrum
- Predicted spectrum for the correct candidate
- Top decoy predicted spectra
- Individual similarity scores

In [ ]:
def plot_spectrum(
    ax,
    mz_bins,
    intensities,
    color="black",
    label=None,
    alpha=1.0,
    invert=False,
):
    """Plot a stick spectrum on the given axes."""
    sign = -1 if invert else 1
    for mz, intens in zip(mz_bins, intensities):
        if intens > 0:
            ax.vlines(
                mz, 0, sign * intens, colors=color, linewidth=0.8, alpha=alpha
            )
    if label:
        ax.plot([], [], color=color, label=label, alpha=alpha)


def inspect_molecule(
    spec_id, df, hdf5_path, similarity_columns, n_top_decoys=5
):
    """Inspect a single target molecule's retrieval results.

     Parameters
    -------
     spec_id : str
         Target identifier (InChIKey for RI match, spec index for formula match).
     df : pd.DataFrame
         Full retrieval results.
     hdf5_path : str
         Path to spectra HDF5 file.
     similarity_columns : list[str]
         Similarity column names.
     n_top_decoys : int
         Number of top-ranked decoys to show.
    """
    spec_rows = df[df["spec"] == spec_id]
    if len(spec_rows) == 0:
        print(f"No results found for spec_id={spec_id}")
        return

    correct = spec_rows[~spec_rows["is_decoy"]]
    decoys = spec_rows[spec_rows["is_decoy"]]

    # Print summary
    print(f"Target: {spec_id}")
    if len(correct) > 0:
        print(f"Correct SMILES: {correct.iloc[0]['smiles']}")
    print(f"Candidates: {len(spec_rows)} ({len(decoys)} decoys)")
    print()

    # Similarity scores table
    score_rows = []
    for sim_col in similarity_columns:
        rank_col = f"rank_{sim_col}"
        row = {"metric": sim_col}
        if len(correct) > 0 and sim_col in correct.columns:
            row["correct_score"] = f"{correct.iloc[0][sim_col]:.4f}"
        if len(correct) > 0 and rank_col in correct.columns:
            row["correct_rank"] = int(correct.iloc[0][rank_col])
        if len(decoys) > 0 and sim_col in decoys.columns:
            best_decoy_idx = decoys[sim_col].idxmax()
            row["best_decoy_score"] = (
                f"{decoys.loc[best_decoy_idx, sim_col]:.4f}"
            )
            row["best_decoy_smiles"] = decoys.loc[best_decoy_idx, "smiles"]
        score_rows.append(row)
    print("Similarity scores:")
    display(pd.DataFrame(score_rows))

    # Plot spectra if HDF5 is available
    if not Path(hdf5_path).exists():
        print("HDF5 not available, skipping spectrum plots")
        return

    with h5py.File(hdf5_path, "r") as hf:
        spec_key = str(spec_id)
        if spec_key not in hf:
            print(f"Target {spec_key} not found in HDF5")
            return

        mz_bins = hf["mz_bins"][:]
        grp = hf[spec_key]
        gt_spec = grp["ground_truth_intensities"][:]
        pred_matrix = grp["candidate_predicted_intensities"][:]
        cand_smiles = [
            s.decode() if isinstance(s, bytes) else s
            for s in grp["candidate_smiles"][:]
        ]
        cand_is_decoy = grp["candidate_is_decoy"][:]

    # Find correct and top decoy indices
    correct_idx = np.where(~cand_is_decoy)[0]
    decoy_indices = np.where(cand_is_decoy)[0]

    # Rank decoys by primary metric
    primary_sim_col = (
        similarity_columns[0] if similarity_columns else "cosine_similarity"
    )
    if primary_sim_col in spec_rows.columns:
        decoy_rows = spec_rows[spec_rows["is_decoy"]].sort_values(
            primary_sim_col, ascending=False
        )
        top_decoy_smiles = decoy_rows["smiles"].head(n_top_decoys).tolist()
        # Map SMILES back to indices in the HDF5 matrix
        smiles_to_idx = {s: i for i, s in enumerate(cand_smiles)}
        top_decoy_indices = [
            smiles_to_idx[s] for s in top_decoy_smiles if s in smiles_to_idx
        ]
    else:
        top_decoy_indices = decoy_indices[:n_top_decoys].tolist()

    # Plot 1: GT vs correct prediction (mirror plot)
    fig, axes = plt.subplots(
        1 + len(top_decoy_indices),
        1,
        figsize=(10, 2.5 * (1 + len(top_decoy_indices))),
        sharex=True,
    )
    if not isinstance(axes, np.ndarray):
        axes = [axes]

    ax = axes[0]
    plot_spectrum(
        ax, mz_bins, gt_spec, color="black", label="Experimental (GT)"
    )
    if len(correct_idx) > 0:
        correct_pred = pred_matrix[correct_idx[0]]
        plot_spectrum(
            ax,
            mz_bins,
            correct_pred,
            color="tab:blue",
            label="Predicted (correct)",
            invert=True,
        )
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_ylabel("Intensity")
    ax.set_title(
        f"Correct candidate: {cand_smiles[correct_idx[0]] if len(correct_idx) > 0 else 'N/A'}"
    )
    ax.legend(fontsize=8, loc="upper right")

    # Plot 2+: GT vs top decoys
    decoy_colors = [
        "tab:red",
        "tab:orange",
        "tab:purple",
        "tab:brown",
        "tab:pink",
    ]
    for i, didx in enumerate(top_decoy_indices):
        ax = axes[1 + i]
        plot_spectrum(
            ax, mz_bins, gt_spec, color="black", label="Experimental (GT)"
        )
        color = decoy_colors[i % len(decoy_colors)]
        plot_spectrum(
            ax,
            mz_bins,
            pred_matrix[didx] / pred_matrix[didx].max(),
            color=color,
            label=f"Decoy #{i + 1}",
            invert=True,
        )
        ax.axhline(0, color="gray", linewidth=0.5)
        ax.set_ylabel("Intensity")
        # Show decoy similarity score
        decoy_smi = cand_smiles[didx]
        decoy_score_str = ""
        if primary_sim_col in spec_rows.columns:
            decoy_row = spec_rows[spec_rows["smiles"] == decoy_smi]
            if len(decoy_row) > 0:
                decoy_score_str = f" ({primary_sim_col}={decoy_row.iloc[0][primary_sim_col]:.4f})"
        ax.set_title(
            f"Decoy #{i + 1}: {decoy_smi[:60]}{'...' if len(decoy_smi) > 60 else ''}{decoy_score_str}"
        )
        ax.legend(fontsize=8, loc="upper right")

    axes[-1].set_xlabel("m/z")
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect a specific molecule by index in the overview table
# Change the index to explore different molecules

# Examples:
# - Row 0 = best-ranked molecule (rank=1 for primary metric)
# - Last rows = worst-ranked molecules (failures)

INSPECT_ROW = 0  # index into df_overview (sorted by rank)

target_spec_id = df_overview.iloc[INSPECT_ROW]["spec"]
inspect_molecule(
    target_spec_id, df, HDF5_PATH, SIMILARITY_COLUMNS, n_top_decoys=3
)

In [ ]:
# Inspect a failure case (worst rank)
INSPECT_ROW_FAILURE = -1  # last row = worst rank

target_spec_id = df_overview.iloc[INSPECT_ROW_FAILURE]["spec"]
inspect_molecule(
    target_spec_id, df, HDF5_PATH, SIMILARITY_COLUMNS, n_top_decoys=3
)

## Summary table: top-k accuracies

A compact table with columns for top-1, 2, 3, ... accuracy, MRR, mean rank, and median rank.

In [ ]:
def retrieval_summary_table(df, similarity_columns, k_values=None):
    """Compute a clean summary table of retrieval metrics.

     Parameters
    -------
     df : pd.DataFrame
         Raw retrieval results (with spec, is_decoy, rank_*, similarity columns).
     similarity_columns : list[str]
         Similarity column names (without 'rank_' prefix).
     k_values : list[int], optional
         Top-k values to report. Defaults to [1, 2, 3, 4, 5, 10, 15, 20, 50].

     Returns
    ----
     pd.DataFrame
         Rows = similarity metrics, columns = top-k accuracies + MRR + mean/median rank.
    """
    if k_values is None:
        k_values = [1, 2, 3, 4, 5, 10, 15, 20, 50]

    rows = []
    for sim_col in similarity_columns:
        rank_col = f"rank_{sim_col}"
        if rank_col not in df.columns:
            continue

        # Get the rank of the correct answer for each target
        correct_ranks = df[~df["is_decoy"]].groupby("spec")[rank_col].min()

        row = {"metric": sim_col}

        # Top-k accuracies
        for k in k_values:
            acc = (correct_ranks <= k).mean() * 100
            row[f"top-{k}"] = f"{acc:.1f}"

        # MRR
        mrr = (1.0 / correct_ranks).mean()
        row["MRR"] = f"{mrr:.4f}"

        # Mean and median rank
        row["mean_rank"] = f"{correct_ranks.mean():.2f}"
        row["median_rank"] = f"{correct_ranks.median():.1f}"

        # N queries
        row["n_queries"] = len(correct_ranks)

        rows.append(row)

    summary = pd.DataFrame(rows)
    summary = summary.set_index("metric")
    return summary


df_summary = retrieval_summary_table(df, SIMILARITY_COLUMNS, k_values=K_VALUES)
display(df_summary)

In [ ]:
def compare_retrieval_results(
    csv_paths, labels=None, similarity_columns=None, k_values=None
):
    """Compare retrieval results from multiple CSV files side by side.

     Parameters
    -------
     csv_paths : list[str]
         Paths to retrieval result CSV files.
     labels : list[str], optional
         Display labels for each file. Defaults to filenames.
     similarity_columns : list[str], optional
         Which similarity columns to report. Auto-detected if None.
     k_values : list[int], optional
         Top-k values. Defaults to [1, 5, 10, 20].

     Returns
    ----
     pd.DataFrame
         Combined summary table.
    """
    if k_values is None:
        k_values = [1, 5, 10, 20]
    if labels is None:
        labels = [Path(p).stem for p in csv_paths]

    all_summaries = []
    for path, label in zip(csv_paths, labels):
        if not Path(path).exists():
            print(f"Skipping {path} (not found)")
            continue
        df_i = pd.read_csv(path)

        # Auto-detect sim columns
        sim_cols = similarity_columns
        if sim_cols is None:
            sim_cols = [
                c.replace("rank_", "")
                for c in df_i.columns
                if c.startswith("rank_")
            ]

        summary = retrieval_summary_table(df_i, sim_cols, k_values=k_values)
        summary["source"] = label
        summary = summary.reset_index()
        all_summaries.append(summary)

    if all_summaries:
        combined = pd.concat(all_summaries, ignore_index=True)
        combined = combined.set_index(["source", "metric"])
        return combined
    return pd.DataFrame()


# Example: compare formula match vs RI match results
# combined = compare_retrieval_results(
#     csv_paths=[
#         "/path/to/retrieval_with_formula_results.csv",
#         "/path/to/retrieval_with_ri_StdNP_results.csv",
#         "/path/to/retrieval_with_ri_StdNP+SemiStdNP_results.csv",
#     ],
#     labels=["Formula", "RI (StdNP)", "RI (StdNP+SemiStdNP)"],
#     k_values=[1, 5, 10, 20],
# )
# display(combined)

## Correct vs best-decoy similarity scatter

In [ ]:
sim_col = PRIMARY_RANKING_METRIC
correct_col = f"correct_{sim_col}"
decoy_col = f"best_decoy_{sim_col}"

if correct_col in df_overview.columns and decoy_col in df_overview.columns:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(
        df_overview[correct_col],
        df_overview[decoy_col],
        alpha=0.4,
        s=10,
        edgecolors="none",
    )
    lims = [0, 1]
    ax.plot(lims, lims, "--", color="gray", linewidth=0.8)
    ax.set_xlabel(f"Correct candidate {sim_col}")
    ax.set_ylabel(f"Best decoy {sim_col}")
    ax.set_title("Correct vs best decoy similarity")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
else:
    print(f"Columns {correct_col} / {decoy_col} not found in overview table")